# Lab-2b: Data Preparation with SageMaker Processing Jobs

**Persona:** Data Engineer &nbsp;|&nbsp; **Lab:** 2 - Data Prep

## Overview

This notebook produces the feature-ready **train** and **test** datasets that the model-build lab (Lab 3) consumes. Instead of preprocessing data inline in the training notebook, we run a **standalone, scalable SageMaker Processing Job** so that data preparation is a reusable, independently schedulable step in the ML workflow.

You'll use the **bank marketing dataset** (UCI Machine Learning Repository) to predict whether a customer will subscribe to a term deposit. The preprocessing steps are:

1. Download the raw dataset and stage it in Amazon S3
2. Run a SageMaker Processing Job (scikit-learn container) that:
   - Label-encodes categorical features
   - Encodes the binary target (`y`)
   - Performs a stratified train/test split
   - Writes `train.csv` and `test.csv` back to S3
3. Record the S3 output locations used downstream by Lab 3

### Why a Processing Job?

**SageMaker Processing Jobs** run your preprocessing script inside a managed container on dedicated compute. SageMaker provisions the instance, pulls the container, runs the script, uploads outputs to S3, and tears the instance down when done. This keeps data prep:
- **Decoupled** from experimentation and training
- **Reproducible** and version-controlled (the script is an artifact)
- **Scalable** to larger datasets and instance types without changing the notebook

### Output contract (consumed by Lab 3)

- `s3://<default-bucket>/bank-marketing-lab/data/train/train.csv`
- `s3://<default-bucket>/bank-marketing-lab/data/test/test.csv`

Each CSV has the **target as the first column** and **no header row**, matching what the XGBoost training script in Lab 3 expects.

---

## Section 1: Setup

Install the SageMaker Python SDK v3 and supporting libraries, then initialize the SageMaker session. The first cell may take 1-2 minutes and will restart the kernel.

In [ ]:
# Install required packages
# This may take 1-2 minutes on first run. Ignore dependency conflict warnings.
!pip install --upgrade pip -q

# Clean uninstall to avoid cached version conflicts
%pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops -q

# Reinstall with compatible versions
%pip install --no-cache-dir "sagemaker>=3.17,<4" \
    "pandas" "scikit-learn" -q

# Restart kernel to pick up updated packages
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
# Import required libraries. Restart kernel if you got stuck in this cell excetion.
import boto3
import sagemaker
import pandas as pd
import os
import io
import importlib.metadata

# SageMaker v3 session helpers
from sagemaker.core.helper.session_helper import Session, get_execution_role

# SageMaker v3 Processing APIs
from sagemaker.core.processing import ScriptProcessor, ProcessingInput, ProcessingOutput, ProcessingS3Input
from sagemaker.core.image_uris import retrieve as retrieve_image_uri
from sagemaker.core.shapes.shapes import ProcessingS3Output

print('✓ All libraries imported successfully')
print(f'SageMaker SDK version: {importlib.metadata.version("sagemaker")}')

In [ ]:
# Initialize SageMaker session and get AWS configuration
import os, sys
sagemaker_session = Session()
region = sagemaker_session.boto_region_name
role = get_execution_role()
bucket = sagemaker_session.default_bucket()

# Data prefix from the single shared source (repo-root .env via shared loader).
for _c in [os.getcwd(), *[str(p) for p in __import__('pathlib').Path(os.getcwd()).parents]]:
    if os.path.exists(os.path.join(_c, 'workshop_env.py')):
        sys.path.insert(0, _c); break
from workshop_env import load_workshop_env
prefix = load_workshop_env()['DATA_PREFIX']

print('AWS Configuration:')
print(f'  Region: {region}')
print(f'  S3 Bucket: {bucket}')
print(f'  IAM Role: {role}')
print(f'  Data Prefix: {prefix}')
print('\n SageMaker session initialized successfully')

---

## Section 2: Acquire Raw Dataset and Stage in S3

A Processing Job reads its inputs from Amazon S3. We first download the raw bank marketing dataset, then upload the raw CSV to S3 so the job can pull it into the container.

### Dataset Features

- **Client Demographics**: age, job type, marital status, education level
- **Financial Profile**: credit default status, housing loans, personal loans
- **Campaign Details**: contact method, timing (month/day), call duration, number of contacts, previous outcomes
- **Economic Indicators**: employment variation rate, consumer price/confidence indices, Euribor rate, employment numbers
- **Target Variable**: binary - did the client subscribe to a term deposit (yes/no)

In [ ]:
# Download and extract the dataset
print('Downloading bank marketing dataset...')
!wget -N https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip
!unzip -o bank-additional.zip
print('\n✓ Dataset downloaded and extracted')

In [ ]:
# Quick look at the raw data (semicolon-separated)
raw_df = pd.read_csv('bank-additional/bank-additional-full.csv', sep=';')
print(f'Raw shape: {raw_df.shape}')
raw_df.head()

In [ ]:
# Upload the raw dataset to S3 as the Processing Job input
raw_data_s3 = sagemaker_session.upload_data(
    'bank-additional/bank-additional-full.csv',
    bucket,
    f'{prefix}/data/raw'
)
print(f'Raw data staged at: {raw_data_s3}')

---

## Section 3: Author the Preprocessing Script

The Processing Job executes this script inside the scikit-learn container. It reads the raw CSV from the input channel, encodes features and the target, performs a stratified split, and writes `train.csv` / `test.csv` to the output channels.

**Container paths** (SageMaker Processing convention):
- Input: `/opt/ml/processing/input`
- Train output: `/opt/ml/processing/output/train`
- Test output: `/opt/ml/processing/output/test`

SageMaker maps these local paths to/from S3 based on the `ProcessingInput` / `ProcessingOutput` configuration in Section 4.

In [ ]:
# Create the preprocessing script that runs inside the Processing Job
os.makedirs('processing', exist_ok=True)

preprocessing_script = '''import argparse
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--test-size', type=float, default=0.2)
    parser.add_argument('--random-state', type=int, default=42)
    return parser.parse_args()


if __name__ == '__main__':
    args = parse_args()

    input_dir = '/opt/ml/processing/input'
    train_dir = '/opt/ml/processing/output/train'
    test_dir = '/opt/ml/processing/output/test'
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    # Load raw data (semicolon-separated)
    input_file = os.path.join(input_dir, 'bank-additional-full.csv')
    df = pd.read_csv(input_file, sep=';')
    print(f'Loaded raw data shape: {df.shape}')

    # Encode categorical features (all object columns except the target)
    cat_cols = [c for c in df.select_dtypes(include=['object']).columns if c != 'y']
    for col in cat_cols:
        df[col] = LabelEncoder().fit_transform(df[col])

    # Encode target
    df['y'] = (df['y'] == 'yes').astype(int)
    print(f'Encoded {len(cat_cols)} categorical features')

    # Stratified train/test split
    X, y = df.drop('y', axis=1), df['y']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=args.test_size, random_state=args.random_state, stratify=y
    )

    # Save with target as the first column and no header (XGBoost CSV convention)
    pd.concat([y_train, X_train], axis=1).to_csv(
        os.path.join(train_dir, 'train.csv'), index=False, header=False
    )
    pd.concat([y_test, X_test], axis=1).to_csv(
        os.path.join(test_dir, 'test.csv'), index=False, header=False
    )

    print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
    print('✓ Preprocessing complete')
'''

with open('processing/preprocessing.py', 'w') as f:
    f.write(preprocessing_script)

print('✓ Preprocessing script created at processing/preprocessing.py')

---

## Section 4: Run the SageMaker Processing Job

Configure a `FrameworkProcessor` (managed scikit-learn 1.4-2 container) and launch the job. We pin the output destinations to the exact S3 locations that Lab 3 reads from, so downstream training requires no changes.

The job takes a few minutes: SageMaker provisions the instance, runs `preprocessing.py`, uploads the outputs to S3, and terminates the instance.

In [ ]:
# Configure the scikit-learn processor using v3 API
# Retrieve the managed scikit-learn container image URI
sklearn_image_uri = retrieve_image_uri(
    'sklearn', region, version='1.2-1', image_scope='training'
)
print(f'Container image: {sklearn_image_uri}')

sklearn_processor = ScriptProcessor(
    role=role,
    image_uri=sklearn_image_uri,
    instance_type='ml.m5.xlarge',
    instance_count=1,
    command=['python3'],
    base_job_name='bank-marketing-preprocess',
    sagemaker_session=sagemaker_session
)

print('✓ ScriptProcessor (SKLearn 1.2-1) configured')

In [ ]:
# Pin output destinations to the locations Lab 3 consumes
train_output_s3 = f's3://{bucket}/{prefix}/data/train'
test_output_s3 = f's3://{bucket}/{prefix}/data/test'

sklearn_processor.run(
    code='processing/preprocessing.py',
    inputs=[
        ProcessingInput(
            input_name='raw-data',
            s3_input=ProcessingS3Input(
                s3_uri=raw_data_s3,
                s3_data_type='S3Prefix',
                local_path='/opt/ml/processing/input',
                s3_input_mode='File'
            )
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name='train',
            s3_output=ProcessingS3Output(
                s3_uri=train_output_s3,
                s3_upload_mode='EndOfJob',
                local_path='/opt/ml/processing/output/train'
            )
        ),
        ProcessingOutput(
            output_name='test',
            s3_output=ProcessingS3Output(
                s3_uri=test_output_s3,
                s3_upload_mode='EndOfJob',
                local_path='/opt/ml/processing/output/test'
            )
        )
    ],
    arguments=['--test-size', '0.2', '--random-state', '42'],
    wait=False,
    logs=False
)

# Wait quietly without the flashing spinner panel
import time
job_name = sklearn_processor.latest_job.processing_job_name
sm_client = sagemaker_session.sagemaker_client
print(f'Processing job: {job_name}')

while True:
    resp = sm_client.describe_processing_job(ProcessingJobName=job_name)
    status = resp['ProcessingJobStatus']
    if status in ('Completed', 'Failed', 'Stopped'):
        break
    print(f'  Status: {status} ...', flush=True)
    time.sleep(30)

if status == 'Completed':
    print('\n✓ Processing job complete')
else:
    print(f'\n✗ Job ended with status: {status}')
    if 'FailureReason' in resp:
        print(f'  Reason: {resp["FailureReason"]}')

---

## Section 5: Verify Outputs

Confirm the train and test datasets landed in S3 and preview the first few rows. These are the exact paths Lab 3 (`lab3a_traditional_ml_experimenation.ipynb`) uses for training and evaluation.

In [ ]:
# Final S3 locations of the prepared datasets
train_s3 = f'{train_output_s3}/train.csv'
test_s3 = f'{test_output_s3}/test.csv'

print('Prepared datasets (used by Lab 3):')
print(f'  Train: {train_s3}')
print(f'  Test:  {test_s3}')

In [ ]:
# Preview the prepared train dataset directly from S3
def preview_s3_csv(s3_uri, n=5):
    _, _, rest = s3_uri.partition('s3://')
    b, _, key = rest.partition('/')
    obj = boto3.client('s3', region_name=region).get_object(Bucket=b, Key=key)
    df = pd.read_csv(io.BytesIO(obj['Body'].read()), header=None)
    print(f'{s3_uri}  ->  shape {df.shape} (col 0 is the target)')
    return df.head(n)


preview_s3_csv(train_s3)

In [ ]:
# Preview the prepared test dataset
preview_s3_csv(test_s3)

---

## Summary

You prepared the bank marketing dataset with a standalone **SageMaker Processing Job** and published feature-ready datasets to S3:

- `s3://<default-bucket>/bank-marketing-lab/data/train/train.csv`
- `s3://<default-bucket>/bank-marketing-lab/data/test/test.csv`

Because these paths are derived from the default bucket and the `bank-marketing-lab` prefix, **Lab 3** picks them up automatically - no manual copy/paste needed.

### Next Step

Run **`lab-2-traditional-ml-iceberg-registration.ipynb`** to register the processed data into S3 Table Bucket (managed Iceberg) and Athena Iceberg tables. This step is shared across Labs 2B–2E and creates the tables that Lab 3 and Lab 5 consume.

### Why this matters

- **Separation of concerns**: data prep is decoupled from training and experimentation
- **Reusability**: the same processing script can be scheduled, reused, or wired into a SageMaker Pipeline
- **Scalability**: scale to larger data by changing instance type/count, not notebook code